In [ ]:
from pathlib import Path  # Portable filesystem paths.
import numpy as np  # Numerical array operations.
import imageio.v3 as imageio  # PNG input and output.
import torch  # Differentiable tensor operations.
import torch.nn.functional as F  # Padding and convolution operations.


# Use underscores in radius folders.
def radius_folder(radius_um):
    return f"{radius_um:g}".replace(".", "_")


# Locate the measurement PNG.
def measurement_path(cls, object_id, radius_um):
    return (
        meas_root
        / cls
        / str(object_id)
        / radius_folder(radius_um)
        / measurement_filename
    )


# Group outputs by object and radius.
def result_directory(cls, object_id, radius_um):
    return (
        output_root
        / cls
        / str(object_id)
        / radius_folder(radius_um)
    )


# Load grayscale values on [0, 1].
def read_png01(path, dtype=np.float64):
    raw = imageio.imread(path)

    # Scale integers by their datatype range.
    if np.issubdtype(raw.dtype, np.integer):
        image = raw.astype(dtype) / np.iinfo(raw.dtype).max
    else:
        image = raw.astype(dtype)

    # Convert colour data to grayscale.
    if image.ndim == 3:
        if image.shape[2] == 1:
            image = image[..., 0]

        elif image.shape[2] in (3, 4):
            image = (
                0.2989 * image[..., 0]
                + 0.5870 * image[..., 1]
                + 0.1140 * image[..., 2]
            )

    # Require finite values within [0, 1].
    if (
        image.ndim != 2
        or not np.isfinite(image).all()
        or image.min() < 0
        or image.max() > 1
    ):
        raise ValueError(
            f"Expected a finite image in [0, 1]: {path}"
        )

    return image


# Prepare the detector measurement.
def load_measurement(path):
    image = read_png01(path, dtype=np.float32)

    # Check detector dimensions.
    if image.shape != (num_px, num_px):
        raise ValueError(
            f"Wrong measurement shape {image.shape}: {path}"
        )

    # Reject constant measurements.
    if np.ptp(image) == 0:
        raise ValueError(f"Constant measurement: {path}")

    # Transfer data to the selected device.
    return torch.from_numpy(image).to(device)


# Move zero frequency to the centre.
def fftshift(x):
    for dim in (-2, -1):
        x = torch.roll(
            x,
            shifts=x.size(dim) // 2,
            dims=dim,
        )
    return x


# Restore standard FFT ordering.
def ifftshift(x):
    for dim in (-2, -1):
        x = torch.roll(
            x,
            shifts=-(x.size(dim) // 2),
            dims=dim,
        )
    return x


# Build the centred spatial grid.
def create_grid_like_numpy(nx, dx, ny, dy):
    x = torch.linspace(
        -dx * nx / 2,
        dx * nx / 2,
        nx,
        device=device,
        dtype=torch.float32,
    )

    y = torch.linspace(
        -dy * ny / 2,
        dy * ny / 2,
        ny,
        device=device,
        dtype=torch.float32,
    )

    return torch.meshgrid(x, y, indexing="xy")


# Compute the coherent propagation kernel.
def precompute_angular_spectrum_kernel():
    # Construct centred frequency indices.
    ix = torch.arange(
        -nx / 2, nx / 2,
        device=device,
        dtype=torch.float32,
    )

    iy = torch.arange(
        -ny / 2, ny / 2,
        device=device,
        dtype=torch.float32,
    )

    # Calculate wavelength-scaled squared frequencies.
    qx = (wavelength / dx_camera / nx * ix) ** 2
    qy = (wavelength / dx_camera / ny * iy) ** 2
    q = qy[:, None] + qx[None, :]

    # Apply the angular-spectrum phase shift.
    H = torch.exp(
        2j * np.pi / wavelength * z_obj_cam
        * torch.sqrt(1.0 - torch.clamp(q, max=1.0))
    )

    # Remove evanescent components.
    H[q > 1.0] = 0.0

    # Arrange the kernel for FFT multiplication.
    return ifftshift(H).to(torch.complex64)


# Calculate the source coherence factor.
def compute_gamma(radius_um):
    # Determine source-plane sampling.
    dx_src = dx_camera * z_source_object / z_obj_cam

    X, Y = create_grid_like_numpy(
        nx, dx_src, ny, dx_src
    )

    # Define a uniform circular source.
    R = torch.sqrt(X ** 2 + Y ** 2)
    source = (R < radius_um * 1e-6).to(torch.float32)

    # Reject an empty sampled source.
    if source.sum().item() == 0:
        raise ValueError(
            f"Empty sampled source at r_s={radius_um:g} um"
        )

    del X, Y, R

    # Take the centred source Fourier transform.
    g = fftshift(torch.fft.fft2(ifftshift(source)))
    # Normalise the coherence factor.
    gamma = g / (g.detach().abs().max() + 1e-12)

    return gamma.to(torch.complex64)


# Apply detached min-max normalisation.
def normalise_prediction(image):
    # Exclude scaling constants from differentiation.
    low = image.detach().amin()
    span = (image.detach().amax() - low).clamp_min(1e-12)
    return (image - low) / span


# Predict the detector intensity.
def forward_intensity(
    amp,
    H_as,
    phase_factor,
    gamma=None,
):
    # Pad with unit amplitude and zero phase.
    field_pad = F.pad(
        amp.clamp(0, 1),
        (pad, pad, pad, pad),
        value=1.0,
    ).to(torch.complex64)

    # Coherent propagation branch.
    if gamma is None:
        U = torch.fft.ifft2(
            torch.fft.fft2(field_pad * phase_factor) * H_as
        )

        intensity = torch.abs(U) ** 2

    # Partially coherent propagation branch.
    else:
        U = torch.fft.ifft2(
            torch.fft.fft2(field_pad * phase_factor) * H_as
        )

        Ic = torch.abs(U) ** 2

        # Apply the Schell coherence operation.
        tmp = fftshift(
            torch.fft.ifft2(ifftshift(Ic))
        ) * gamma

        intensity = torch.abs(
            fftshift(torch.fft.fft2(ifftshift(tmp)))
        )

    # Remove propagation padding.
    crop = intensity[
        pad:pad + num_px,
        pad:pad + num_px,
    ]

    # Normalise the predicted measurement.
    return normalise_prediction(crop.to(torch.float32))


# Compute differentiable structural similarity.
def ssim_torch(img1, img2, window_size=11, sigma=1.5):
    # Build a Gaussian averaging window.
    coords = torch.arange(
        window_size,
        device=img1.device,
        dtype=torch.float32,
    ) - (window_size - 1) / 2.0

    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / (g.sum() + 1e-12)

    w = torch.outer(g, g)
    w = (w / (w.sum() + 1e-12))[None, None]

    # Add batch and channel dimensions.
    x = img1[None, None]
    y = img2[None, None]

    # Calculate Gaussian-weighted local averages.
    def smooth(a):
        return F.conv2d(
            a, w,
            padding=window_size // 2,
        )

    # Calculate local means.
    mu_x = smooth(x)
    mu_y = smooth(y)

    # Calculate local variances and covariance.
    sx = smooth(x * x) - mu_x * mu_x
    sy = smooth(y * y) - mu_y * mu_y
    sxy = smooth(x * y) - mu_x * mu_y

    # Use SSIM constants for unit-range images.
    numerator = (
        (2 * mu_x * mu_y + 0.01 ** 2)
        * (2 * sxy + 0.03 ** 2)
    )

    denominator = (
        (mu_x ** 2 + mu_y ** 2 + 0.01 ** 2)
        * (sx + sy + 0.03 ** 2)
    )

    # Average the SSIM map.
    return (numerator / (denominator + 1e-12)).mean()


# Optimise amplitude with zero object phase.
def reconstruct_one(
    measurement,
    H_as,
    phase_factor,
    gamma,
    tag,
):
    # Start from uniform amplitude 0.5.
    v = torch.full(
        (num_px, num_px),
        0.5,
        device=device,
        dtype=torch.float32,
        requires_grad=True,
    )

    # Configure the Adam optimiser.
    optimizer = torch.optim.Adam([v], lr=lr)

    for iteration in range(1, n_iters + 1):
        # Clear accumulated gradients.
        optimizer.zero_grad()

        # Enforce amplitude bounds.
        amp = v.clamp(0, 1)

        # Generate the detector prediction.
        pred = forward_intensity(
            amp, H_as, phase_factor, gamma
        )

        # Evaluate detector-intensity fitting terms.
        mse = ((pred - measurement) ** 2).mean()
        ssim_value = ssim_torch(pred, measurement)

        # Calculate horizontal amplitude differences.
        dx_tv = F.pad(
            amp[:, 1:] - amp[:, :-1],
            (0, 1, 0, 0),
        )

        # Calculate vertical amplitude differences.
        dy_tv = F.pad(
            amp[1:, :] - amp[:-1, :],
            (0, 0, 0, 1),
        )

        # Compute the mean isotropic TV penalty.
        tv = torch.sqrt(
            dx_tv ** 2 + dy_tv ** 2 + eps_tv ** 2
        ).mean()

        # Combine data fitting and TV.
        loss = (
            w_mse * mse
            + w_ssim * (1 - ssim_value)
            + lambda_tv * tv
        )

        # Stop if optimisation becomes nonfinite.
        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Nonfinite loss: {tag}, iteration {iteration}"
            )

        # Differentiate the reconstruction objective.
        loss.backward()
        # Update the amplitude estimate.
        optimizer.step()

        # Project updated amplitudes onto [0, 1].
        with torch.no_grad():
            v.clamp_(0, 1)

        # Report occasional optimisation progress.
        if (
            iteration == 1
            or iteration % 20 == 0
            or iteration == n_iters
        ):
            print(
                f"[{tag}] {iteration:04d}/{n_iters} | "
                f"loss={loss.item():.6e}",
                flush=True,
            )

    # Return the reconstructed amplitude array.
    return v.detach().clamp(0, 1).cpu().numpy()


# Replace every placeholder before running.
meas_root = Path("REPLACE_WITH_AMPLITUDE_MEASUREMENT_FOLDER")  # Measurement root folder.
output_root = Path("REPLACE_WITH_OUTPUT_FOLDER")  # Reconstruction output folder.
measurement_filename = "REPLACE_WITH_MEASUREMENT_FILENAME"  # PNG filename within each radius folder.

object_groups = ...  # List of object-class folder names.
object_ids = ...  # List of object IDs to reconstruct.
source_radii_um = ...  # List of source radii (um).

wavelength = ...  # Illumination wavelength (m).
z_source_object = ...  # Source-to-object distance (m).
z_obj_cam = ...  # Object-to-detector distance (m).
dx_camera = ...  # Detector pixel pitch (m).

num_px = ...  # Pixels along each image dimension.
pad_factor = ...  # Integer multiples of num_px, per side.
pad = pad_factor * num_px  # Padding width per side (pixels).
ny = nx = num_px + 2 * pad  # Padded propagation-grid dimensions.
ROI_BORDER = ...  # Border removed from saved images (pixels).

n_iters = ...  # Number of optimisation iterations.
lr = ...  # Adam learning rate.
w_mse = ...  # Mean-squared-error weight.
w_ssim = ...  # Structural-similarity loss weight.
lambda_tv = ...  # Total-variation regularisation weight.
eps_tv = 1e-8  # Smooth the TV gradient at zero.

# Use CUDA when available.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# Save the central amplitude crop.
def save_crop(path, image, border=ROI_BORDER):
    # Move tensor data to NumPy.
    if torch.is_tensor(image):
        image = image.detach().cpu().numpy()

    # Remove the specified border.
    if border:
        crop = image[border:-border, border:-border]
    else:
        crop = image

    # Save 8-bit values without contrast stretching.
    imageio.imwrite(
        path,
        (np.clip(crop, 0, 1) * 255.0 + 0.5).astype(np.uint8),
    )


# Reconstruct each measurement with both models.
def main():
    print("Device:", device)
    output_root.mkdir(parents=True, exist_ok=True)

    # Precompute the propagation kernel once.
    H_as = precompute_angular_spectrum_kernel()

    X, Y = create_grid_like_numpy(
        nx, dx_camera, ny, dx_camera
    )

    # Calculate the quadratic illumination phase.
    phase_factor = torch.exp(
        1j * np.pi / (wavelength * z_source_object)
        * (X ** 2 + Y ** 2)
    ).to(torch.complex64)

    del X, Y

    for radius_um in source_radii_um:
        # Reuse coherence for this source radius.
        gamma = compute_gamma(radius_um)

        # Process all requested classes and objects.
        for cls in object_groups:
            for object_id in object_ids:
                directory = result_directory(
                    cls, object_id, radius_um
                )
                directory.mkdir(parents=True, exist_ok=True)

                # Load the common detector measurement.
                measured = load_measurement(
                    measurement_path(cls, object_id, radius_um)
                )

                # Run coherent and partially coherent inversion.
                for model, model_gamma in (
                    ("COH", None),
                    ("GVS", gamma),
                ):
                    tag = (
                        f"{cls}/{object_id} | "
                        f"r_s={radius_um:g} um | {model}"
                    )

                    # Optimise an independent amplitude estimate.
                    reconstruction = reconstruct_one(
                        measured,
                        H_as,
                        phase_factor,
                        model_gamma,
                        tag,
                    )

                    # Save each model using identical cropping.
                    save_crop(
                        directory / f"amp_{model}_CROP.png",
                        reconstruction,
                        border=ROI_BORDER,
                    )
                    del reconstruction

                # Release temporary measurement references.
                del measured, model_gamma

        del gamma

    del H_as, phase_factor

    # Release unused GPU cache.
    if device.type == "cuda":
        torch.cuda.empty_cache()

    print("Done. Reconstructions saved under:", output_root.resolve())


# Run the reconstruction workflow.
if __name__ == "__main__":
    main()
